# GeoMind AI: Feature Engineering & Preprocessing Pipeline

**Target Role:** Amazon Applied Scientist I (Intern)
**Objective:** Build, validate, and visualize the feature engineering transformations and leakage-safe ColumnTransformer pipeline for next-hour ($t+1$) traffic forecasting.

---
### Topics Covered:
1. **Target Formulation ($t+1$):** Creating `future_traffic_volume` with temporal gap protection.
2. **Cyclical Temporal Encoding:** Mapping hour and day_of_week into continuous circle ($\sin, \cos$).
3. **Autoregressive Lags:** Capturing short-term and diurnal memory ($	ext{lag}_1$ to $	ext{lag}_{24}$).
4. **Rolling Statistics:** Closed-left moving averages and standard deviations.
5. **Leakage-Safe Preprocessing:** Fitting scikit-learn `ColumnTransformer` strictly on `train.csv`.

In [ ]:
import os
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Add project root to sys.path
module_path = str(Path(os.getcwd()).parent if 'notebooks' in os.getcwd() else Path(os.getcwd()))
if module_path not in sys.path:
    sys.path.insert(0, module_path)

from src.feature_engineering import engineer_features, create_target_variable
from src.data_preprocessing import TrafficPreprocessor, prepare_datasets

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (12, 5)
print('Imports successful!')

## 1. Target Formulation ($t+1$) & Gap Filtering
In real sensor data, rows can be separated by multi-hour gaps due to hardware downtime. A naive `df.shift(-1)` pairs a Friday night record with a Monday morning record. We verify that $t_{i+1} - t_i == 1\text{ hr}$.

In [ ]:
train_raw = pd.read_csv('../data/processed/train.csv')
train_raw['date_time'] = pd.to_datetime(train_raw['date_time'])

train_with_target = create_target_variable(train_raw)
print(f'Total observations: {len(train_with_target):,}')
print(f'Valid continuous next-hour targets: {train_with_target["future_traffic_volume"].notnull().sum():,}')
train_with_target[['date_time', 'traffic_volume', 'future_traffic_volume']].head(6)

## 2. Cyclical Temporal Encodings
Hour 23 and Hour 0 are 1 hour apart in reality, but $|23 - 0| = 23$ in integer space. Trigonometric projection maps them continuously onto a 2D circle:
$$\sin\left(\frac{2\pi \cdot \text{hour}}{24}\right), \quad \cos\left(\frac{2\pi \cdot \text{hour}}{24}\right)$$

In [ ]:
hours = np.arange(24)
sin_h = np.sin(2 * np.pi * hours / 24)
cos_h = np.cos(2 * np.pi * hours / 24)

plt.figure(figsize=(6, 6))
plt.scatter(sin_h, cos_h, c=hours, cmap='twilight', s=100)
for h, x, y in zip(hours, sin_h, cos_h):
    plt.annotate(f'{h}h', (x*1.12, y*1.12), ha='center', fontsize=9)
plt.title('Cyclical 24-Hour Trigonometric Projection', fontsize=13, fontweight='bold')
plt.xlabel('sin(hour)')
plt.ylabel('cos(hour)')
plt.axis('equal')
plt.grid(True, alpha=0.3)
plt.show()

## 3. Autoregressive Lag & Rolling Feature Engineering
We run the full feature generation pipeline on `train_raw`.

In [ ]:
train_feat = engineer_features(train_raw)
print(f'Engineered train shape: {train_feat.shape}')
lag_cols = [c for c in train_feat.columns if 'lag' in c or 'rolling' in c]
train_feat[['date_time', 'traffic_volume', 'future_traffic_volume'] + lag_cols[:4]].head()

## 4. Feature Correlation with Target ($y = \text{future\_traffic\_volume}$)
Which engineered features correlate most strongly with next-hour traffic volume?

In [ ]:
num_cols = train_feat.select_dtypes(include=[np.number]).columns
corrs = train_feat[num_cols].corr()['future_traffic_volume'].drop('future_traffic_volume').sort_values(ascending=False)

plt.figure(figsize=(10, 8))
corrs.plot(kind='barh', color='#1f77b4')
plt.title('Feature Correlation with Next-Hour Traffic Volume (t+1)', fontsize=13, fontweight='bold')
plt.xlabel('Pearson Correlation Coefficient (r)')
plt.grid(True, alpha=0.3)
plt.show()

print('Top 5 Positively Correlated Features:')
print(corrs.head(5))
print('\nTop 5 Negatively Correlated Features:')
print(corrs.tail(5))

## 5. End-to-End Leakage-Safe Dataset Preparation
Fitting the `ColumnTransformer` strictly on `train.csv` and transforming `val.csv` and `test.csv`.

In [ ]:
X_train, y_train, X_val, y_val, X_test, y_test, feat_names = prepare_datasets()

print('=' * 60)
print('Dataset Preprocessing Completed:')
print(f'  X_train: {X_train.shape} | y_train: {y_train.shape}')
print(f'  X_val:   {X_val.shape}   | y_val:   {y_val.shape}')
print(f'  X_test:  {X_test.shape}  | y_test:  {y_test.shape}')
print(f'  Total Features: {len(feat_names)}')
print('=' * 60)